# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mishellscripts/flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

A page is worth reviewing if its decline value (high trend decrease and impressions) is high and it has a low chance of recovery (average position, disregarding time series feature data) unless action is taken. Reason codes it can output are:
- high_decline_low_recovery - top priority and action is needed
- high_decline_likely_recovery - top priority but may self-correct
- low_decline_low_recovery - lower priority, not likely to self-correct
- low_decline_likely_recovery - lower priority, may self-correct
- no_decline - excluded and ranked lowest

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # move from notebooks/ to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [5]:

import numpy as np
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

is_declining          = df["trend_direction"] == "down"
severe_trend           = df["trend_pct"] <= -40
meaningful_impressions = df["impressions_90d"] >= df["impressions_90d"].quantile(0.75)   # top quartile among declining pages
high_decline            = is_declining & severe_trend & meaningful_impressions
low_decline              = is_declining & ~(severe_trend & meaningful_impressions)

recovery_likely = df["avg_position"].between(1, 10)
recovery_low    = ~recovery_likely

def reason_code(declining, high, recov_likely):
    if not declining:
        return "no_decline"
    if high and not recov_likely:
        return "high_decline_low_recovery"
    if high and recov_likely:
        return "high_decline_likely_recovery"
    if not high and not recov_likely:
        return "low_decline_low_recovery"
    return "low_decline_likely_recovery"

df["reason_code"] = [
    reason_code(d, h, r) for d, h, r in zip(is_declining, high_decline, recovery_likely)
]

recovery_penalty = np.where(recovery_low, 2, 1)
df["score"] = np.where(
    is_declining,
    df["trend_pct"].abs() * df["impressions_90d"] * recovery_penalty,
    0
)

queue = df.sort_values("score", ascending=False)[
    ["content_id", "score", "reason_code", "trend_pct", "impressions_90d", "clicks_90d",
     "avg_position", "trend_direction", "content_type"]
]

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(queue)} rows to work/outputs/baseline_action_score.csv")
print(df["reason_code"].value_counts())

Wrote 30000 rows to work/outputs/baseline_action_score.csv
reason_code
no_decline                      13738
low_decline_low_recovery         7633
low_decline_likely_recovery      6075
high_decline_low_recovery        1347
high_decline_likely_recovery     1207
Name: count, dtype: int64


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [6]:
top20 = queue.head(20)
print(top20.to_string(index=False))

          content_id      score                  reason_code  trend_pct  impressions_90d  clicks_90d  avg_position trend_direction    content_type
content_66b4046cc144 38830319.0    high_decline_low_recovery      -89.3           217415          71          26.6            down keyword article
content_5fe46e04994d 23193632.0 high_decline_likely_recovery      -44.8           517715         741           4.2            down keyword article
content_8c19996aa890 22661714.0 high_decline_likely_recovery      -44.5           509252         785           2.5            down keyword article
content_551fe371f51b 19475709.8    high_decline_low_recovery      -84.1           115789          47          23.8            down keyword article
content_124763d39ca5 18977198.6    high_decline_low_recovery      -73.1           129803          17          33.2            down keyword article
content_8b36799b7e44 17731560.0    high_decline_low_recovery      -62.7           141400          23          32.0    

1. content_66b4046cc144
    - Reason Code: high_decline_low_recovery
    - Action: Flag
    - Confidence: High
    - Could be a temporary algorithm update
2. content_5fe46e04994d
    - Reason Code: high_decline_likely_recovery
    - Action: Monitor
    - Confidence: Moderate
    - Could fail to self-recover
3. content_8c19996aa890
    - Reason Code: high_decline_likely_recovery
    - Action: Monitor
    - Confidence: Moderate
    - Could fail to self-recover
4. content_551fe371f51b
    - Reason Code: high_decline_low_recovery
    - Action: Flag
    - Confidence: High
    - Could be a temporary algorithm update
5. content_124763d39ca5
    - Reason Code: high_decline_low_recovery
    - Action:	Flag
    - Confidence: High
    - Only 17 clicks so this could be a low-value page inflated by impressions
6. content_8b36799b7e44
    - Reason Code: high_decline_low_recovery
    - Action: Flag
    - Confidence: High
    - Same as 5, could be low-value
7. content_370de6e8e035
    - Reason Code: high_decline_low_recovery
    - Action:	Flag
    - Confidence: High
    - Could be a temporary algorithm update
8. content_813e88069237
    - Reason Code: low_decline_low_recovery
    - Action: Recheck
    - Confidence: Low
    - Misses decline cutoff by 6% but still high impressions. Reason code could be understated
9. content_4c36c775b818
    - Reason Code: low_decline_likely_recovery
    - Action: Recheck
    - Confidence: Low
    - Missed decline by almost 7% but still high impressions. Reason could be understated
10. content_8d3971bfd976
    - Reason Code: high_decline_low_recovery
    - Action: Flag
    - Confidence: High
    - Low clicks could indicate low-value page
11. content_05e9b4cd9ccf
    - Reason Code: high_decline_low_recovery
    - Action: Flag
    - Confidence: High
    - Could be a temporary algorithm update
12. content_ff94c9b6b411
    - Reason Code: low_decline_low_recovery
    - Action: Monitor
    - Confidence: Low
    - High impressions despite lower decline
13. content_54baba704595
    - Reason Code: high_decline_low_recovery
    - Action: Flag
    - Confidence: Low
    - Only 8 clicks_90d despite ranking in top 20
14. content_fb4bf6555c79
    - Reason Code: high_decline_low_recovery
    - Action: Flag
    - Confidence: Low
    - Only 3 clicks_90d despite ranking in top 20
15. content_e752a4e03dd3
    - Reason Code: high_decline_low_recovery
    - Action: Flag
    - Confidence: High
    - Could be a temporary algorithm update
16. content_b51e2e4d22ff
    - Reason Code: high_decline_low_recovery
    - Action: Flag
    - Confidence: High
    - Could be a temporary algorithm update
17. content_150f89b1d73b
    - Reason Code: high_decline_low_recovery
    - Action: Flag
    - Confidence: High
    - Could be a temporary algorithm update
18. content_cb112fce36be
    - Reason Code: high_decline_likely_recovery
    - Action: Monitor
    - Confidence: Moderate
    - Could fail to self-recover
19. content_2c2606c5d176
    - Reason Code: low_decline_likely_recovery
    - Action: Low
    - Confidence: Low
    - Same threshold/code mismatch
20. content_acf71f98ada2
    - Reason Code: high_decline_low_recovery
    - Action: Flag
    - Confidence: High
    - Could be a temporary algorithm update


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

4 of the top 20 rows are labeled `low_decline` yet outrank some `high_decline` rows by score. This occurs because the trend_pct cutoff was hardcoded shows a problem with using transparent rules in design.

2 rows had clicks < 10 despite ranking high in score. Pages were ranked purely on visibility and can be overstate the cost of a decline.

The score uses `trend_direction`, `trend_pct`, `impressions_90d`, and
`avg_position` only. `is_declining_label` is not an input — `trend_direction`/
`trend_pct` are used in their normal present-tense role, consistent with the rest of this project's leakage rule. No future-window or warehouse fields were involved.



## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.